# 12.12 Temporal & heterogeneous graphs

Time and type are part of the graph, not metadata to discard. Temporal and heterogeneous graphs make time and type part of the signal rather than metadata to discard. This walkthrough builds typed matrices, time-respecting features, and a leakage check across one graph ladder. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 1215
random.seed(SEED)
np.random.seed(SEED)


def make_d1_graph():
    graph = nx.Graph()
    graph.add_nodes_from(range(4))
    graph.add_edges_from([(0, 1), (0, 2), (1, 2), (2, 3)])
    labels = {0: 0, 1: 0, 2: 1, 3: 1}
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_karate_rung():
    graph = nx.karate_club_graph()
    labels = {}
    for node, data in graph.nodes(data=True):
        labels[node] = 0 if data.get("club") == "Mr. Hi" else 1
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_sbm_rung(sizes, p_in, p_out, seed, noise_edges=0):
    probs = []
    for row in range(len(sizes)):
        prob_row = []
        for col in range(len(sizes)):
            prob_row.append(p_in if row == col else p_out)
        probs.append(prob_row)
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    for _ in range(noise_edges):
        u = int(rng.choice(nodes))
        v = int(rng.choice(nodes))
        if u != v:
            graph.add_edge(u, v)
    labels = {}
    offset = 0
    for block, size in enumerate(sizes):
        for node in range(offset, offset + size):
            labels[node] = block
        offset = offset + size
    positions = nx.spring_layout(graph, seed=seed)
    return graph, labels, positions


def make_cora_like_rung():
    graph, labels, positions = make_sbm_rung([18, 16, 14], 0.26, 0.035, SEED + 4, noise_edges=8)
    rng = np.random.default_rng(SEED + 44)
    years = {}
    node_types = {}
    for node in graph.nodes():
        years[node] = int(2016 + rng.integers(0, 7))
        node_types[node] = "paper"
    nx.set_node_attributes(graph, years, "year")
    nx.set_node_attributes(graph, node_types, "kind")
    return graph, labels, positions


def make_large_noisy_rung():
    graph, labels, positions = make_sbm_rung([35, 32, 30, 28], 0.18, 0.045, SEED + 5, noise_edges=55)
    rng = np.random.default_rng(SEED + 55)
    for u, v in graph.edges():
        graph[u][v]["time"] = int(rng.integers(0, 10))
        graph[u][v]["relation"] = "strong" if labels[u] == labels[v] else "weak"
    return graph, labels, positions


def build_graph_ladder():
    d1_graph, d1_labels, d1_pos = make_d1_graph()
    d2_graph, d2_labels, d2_pos = make_karate_rung()
    d3_graph, d3_labels, d3_pos = make_sbm_rung([12, 12, 12], 0.32, 0.04, SEED + 3, noise_edges=5)
    d4_graph, d4_labels, d4_pos = make_cora_like_rung()
    d5_graph, d5_labels, d5_pos = make_large_noisy_rung()
    return [
        {"name": "D1 toy", "graph": d1_graph, "labels": d1_labels, "pos": d1_pos},
        {"name": "D2 karate", "graph": d2_graph, "labels": d2_labels, "pos": d2_pos},
        {"name": "D3 SBM", "graph": d3_graph, "labels": d3_labels, "pos": d3_pos},
        {"name": "D4 Cora-like subset", "graph": d4_graph, "labels": d4_labels, "pos": d4_pos},
        {"name": "D5 larger noisy", "graph": d5_graph, "labels": d5_labels, "pos": d5_pos},
    ]


def partition_from_labels(labels):
    groups = defaultdict(set)
    for node, label in labels.items():
        groups[label].add(node)
    return list(groups.values())


def accuracy_score(y_true, y_pred):
    correct = 0
    total = 0
    for node, truth in y_true.items():
        if node in y_pred:
            correct = correct + int(y_pred[node] == truth)
            total = total + 1
    return correct / max(total, 1)


## The concept, built once (D1): typed matrices and temporal deltas

A temporal edge is $(u,v,t)$ and a heterogeneous graph stores typed matrices such as $A^{\text{user,item}}$. The lesson checks $\Delta_{0,2}=1$, total directed change $2$, $\exp(-0.5)=0.607$, shapes $3\times2$ and $3\times3$, and U-I-U counts $1$ and $0$.

In [ ]:

def directed_snapshot_matrix(edges, n_nodes, cutoff):
    matrix = np.zeros((n_nodes, n_nodes), dtype=int)
    for u, v, time, relation in edges:
        if time <= cutoff:
            matrix[u, v] = 1
    return matrix


def typed_adjacency(edges, users, items, relation_name):
    matrix = np.zeros((len(users), len(items)), dtype=int)
    user_index = {node: index for index, node in enumerate(users)}
    item_index = {node: index for index, node in enumerate(items)}
    for u, v, time, relation in edges:
        if relation == relation_name:
            matrix[user_index[u], item_index[v]] = 1
    return matrix


## Compose meta-path and decay features

Multiplying typed matrices creates meta-path counts; masking by cutoff prevents future edges from entering validation neighborhoods.

In [ ]:

def temporal_hetero_features():
    temporal_edges = [
        (0, 1, 1, "follows"),
        (1, 0, 1, "follows"),
        (0, 2, 3, "follows"),
        (2, 0, 3, "follows"),
    ]
    before = directed_snapshot_matrix(temporal_edges, 4, cutoff=2)
    after = directed_snapshot_matrix(temporal_edges, 4, cutoff=3)
    delta = after - before
    decay = math.exp(-0.5 * 1)
    ui_edges = [
        (0, 0, 1, "click"),
        (1, 0, 1, "click"),
        (2, 1, 1, "click"),
    ]
    user_item = typed_adjacency(ui_edges, [0, 1, 2], [0, 1], "click")
    user_user = np.array(
        [
            [0, 1, 0],
            [1, 0, 0],
            [0, 0, 0],
        ]
    )
    meta_path = user_item @ user_item.T
    return delta, decay, user_item, user_user, meta_path


def make_temporal_edges(graph, labels, seed, relation_noise=0.15):
    rng = np.random.default_rng(seed)
    temporal_edges = []
    for u, v in graph.edges():
        time = int(rng.integers(0, 10))
        same = labels.get(u, 0) == labels.get(v, 1)
        relation = "strong" if same else "weak"
        if rng.random() < relation_noise:
            relation = "weak" if relation == "strong" else "strong"
        temporal_edges.append((u, v, time, relation))
        temporal_edges.append((v, u, time, relation))
    return temporal_edges


delta, decay, user_item, user_user, meta_path = temporal_hetero_features()
print("delta[0,2] =", delta[0, 2])
print("total directed change =", int(np.abs(delta).sum()))
print("decay(age=1) =", round(decay, 3))
print("user-item shape =", user_item.shape)
print("user-user shape =", user_user.shape)
print("U-I-U counts M[0,1], M[0,2] =", meta_path[0, 1], meta_path[0, 2])
assert delta[0, 2] == 1
assert int(np.abs(delta).sum()) == 2
assert round(decay, 3) == 0.607
assert user_item.shape == (3, 2)
assert user_user.shape == (3, 3)
assert meta_path[0, 1] == 1
assert meta_path[0, 2] == 0


## The dataset ladder

D1 is a toy temporal graph; D2-D5 reuse the F11 graph ladder with synthetic timestamps and relation types.

In [ ]:

ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    labels = rung["labels"]
    sample_nodes = list(graph.nodes())[:5]
    sample_edges = list(graph.edges())[:5]
    print(rung["name"])
    print("  nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "classes", sorted(set(labels.values())))
    print("  sample nodes", sample_nodes)
    print("  sample edges", sample_edges)


## Run the same method across D1-D5

In [ ]:

def temporal_neighbor_predict(graph, labels, edges, cutoff):
    predictions = {}
    for node in graph.nodes():
        votes = []
        for u, v, time, relation in edges:
            if v == node and time <= cutoff:
                weight = math.exp(-0.25 * (cutoff - time))
                if relation == "strong":
                    votes.extend([labels[u]] * max(1, int(round(2 * weight))))
                else:
                    votes.append(labels[u])
        if votes:
            predictions[node] = Counter(votes).most_common(1)[0][0]
        else:
            predictions[node] = Counter(labels.values()).most_common(1)[0][0]
    return predictions


ladder = build_graph_ladder()
temporal_results = []
for index, rung in enumerate(ladder):
    graph = rung["graph"]
    labels = rung["labels"]
    edges = make_temporal_edges(graph, labels, SEED + index)
    predictions = temporal_neighbor_predict(graph, labels, edges, cutoff=6)
    score = accuracy_score(labels, predictions)
    rung["temporal_edges"] = edges
    rung["predictions"] = predictions
    rung["accuracy"] = score
    temporal_results.append(score)
    print(f"{rung['name']}: nodes={graph.number_of_nodes()} typed_edges={len(edges)} accuracy={score:.3f}")


## Results visualization

Top row: prediction panels with darker recent edges. Bottom left: time-respecting node-accuracy curve.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for index, rung in enumerate(ladder):
    graph = rung["graph"]
    positions = rung["pos"]
    predictions = rung["predictions"]
    node_colors = [predictions[node] for node in graph.nodes()]
    recent_edges = []
    old_edges = []
    for u, v in graph.edges():
        times = [time for a, b, time, rel in rung["temporal_edges"] if {a, b} == {u, v}]
        max_time = max(times) if times else 0
        if max_time >= 6:
            recent_edges.append((u, v))
        else:
            old_edges.append((u, v))
    nx.draw_networkx_nodes(graph, positions, node_color=node_colors, cmap="tab10", node_size=55, ax=axes[0, index])
    nx.draw_networkx_edges(graph, positions, edgelist=old_edges, edge_color="#dddddd", width=0.6, ax=axes[0, index])
    nx.draw_networkx_edges(graph, positions, edgelist=recent_edges, edge_color="#444444", width=1.2, ax=axes[0, index])
    axes[0, index].set_title(rung["name"])
    axes[0, index].axis("off")
axes[1, 0].plot(range(1, 6), temporal_results, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("node accuracy")
axes[1, 0].set_title("time-respecting accuracy")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung: time leakage

If edges after prediction time enter $A$, validation can look excellent for the wrong reason. Compare leaky and cutoff-masked neighborhoods.

In [ ]:

rung = ladder[-1]
graph = rung["graph"]
labels = rung["labels"]
edges = rung["temporal_edges"]
leaky_predictions = temporal_neighbor_predict(graph, labels, edges, cutoff=9)
masked_predictions = temporal_neighbor_predict(graph, labels, edges, cutoff=6)
leaky_accuracy = accuracy_score(labels, leaky_predictions)
masked_accuracy = accuracy_score(labels, masked_predictions)
future_edges = sum(1 for u, v, time, rel in edges if time > 6)
print("future edges incorrectly available", future_edges)
print("leaky validation accuracy", round(leaky_accuracy, 3))
print("time-masked validation accuracy", round(masked_accuracy, 3))
print("Fix: build every neighborhood with time <= prediction cutoff.")



## Evaluate it + practice

- Metric: compare the rung's node accuracy against the no-skill baseline `majority class` on D1-D5.
- Sanity check: rerun with the fixed seed and verify D1 reproduces the asserted lesson numbers before trusting larger rungs.
- Ablation: include post-cutoff edges, which should inflate validation; the metric should drop or the failure should become visible.
- Failure signal: a high score with a violated split, symmetry, or relation contract is not a real win.
- Inspect the hardest rung by plotting both the graph artifact and the metric curve rather than reading one scalar alone.

Practice prompts:
1. Change only the D3 noise level and predict how the metric curve should move.


2. Replace the D5 seed with another fixed seed and compare the artifact panel.

3. Add one cheap assertion that would catch the lesson pitfall before the metric is printed.